In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

#Data Load

#Download dataset directly from github (note this dataset is originally from Kagle but was downloaded to the github repo to faciliate data loading)
!wget -N 'https://raw.githubusercontent.com/andanovais/machine-learning-zoomcamp-homework/refs/heads/main/07-midterm-project/games_dataset.csv'

#Download dataset directly from github (note this dataset is originally from Kagle but was downloaded to the github repo to faciliate data loading)
df = pd.read_csv('games_dataset.csv')
df.head()

--2026-11-15 18:01:25--  https://raw.githubusercontent.com/andanovais/machine-learning-zoomcamp-homework/refs/heads/main/07-midterm-project/games_dataset.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 48627286 (46M) [text/plain]
Saving to: ‘games_dataset.csv’

games_dataset.csv   100%[===================>]  46.37M   187MB/s    in 0.2s    

Last-modified header missing -- time-stamps turned off.
2026-11-15 18:01:25 (187 MB/s) - ‘games_dataset.csv’ saved [48627286/48627286]



,AppID,Name,Release date,Estimated owners,Required age,Price,DiscountDLC count,Supported languages,Support url,Windows,...,Recommendations,Average playtime forever,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Categories,Genres,Tags,Movies
0,Black Dragon Mage Playtest,"Aug 1, 2023",0 - 0,0,0.00,0,0,[],NaN,True,...,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN
1,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,5.24,65,0,['English'],http://mangagamer.com,True,...,231,8,0,8,0,minori,"Single-player,Steam Trading Cards,Steam Cloud,...",Adventure,"Adventure,Visual Novel,Anime,Cute",NaN
2,Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000,0,4.99,0,0,"['English', 'French', 'German', 'Russian']",https://www.facebook.com/8FloorGames,True,...,0,0,0,0,0,Somer Games,"Single-player,Family Sharing",Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec...",NaN
3,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0 - 20000,1,8.99,0,1,['Korean'],NaN,True,...,0,0,0,0,0,유진게임즈,"Single-player,Steam Achievements,Family Sharing","Casual,Indie,Simulation",NaN,NaN
4,Maze Quest VR,"Apr 24, 2025",0 - 20000,0,4.99,0,0,['English'],https://www.realityexpanded.com,True,...,0,0,0,0,0,Reality Expanded LLC,"Single-player,VR Only,Steam Leaderboards,Famil...","Action,Early Access",NaN,NaN


In [2]:
#------------------DATA PREPARATION AND CLEANING---------

#----Update column header names---------
#convert column header to lower case
df.columns = [str(col).lower() for col in df.columns]

#convert colummn header to replace blank space with _
df.columns = [col.strip().replace(" ", "_") for col in df.columns]

#remove parenthesis from column name
df.columns = [col.strip().replace("(", "") for col in df.columns]
df.columns = [col.strip().replace(")", "") for col in df.columns]

#----Remove data or columns not relevant to model (english supported games for windows players)---------
#Remove columns that are not relevant for intended gaming prediction model
df.drop(['support_url','movies'], axis=1, inplace=True)

#Model focused on windows players. Remove games not for windows.
print(f'Removing non-windows games.... Current count: {df[(df['windows']!=True)].shape[0]}')
df.drop(df[(df['windows']!=True)].index, axis=0, inplace=True) #Drop non-windows games
print(f'Updated Windows Games count: {df[(df['windows']!=True)].shape[0]}')
#Remove column for OS since now only windows games are being considered.
df.drop(['windows','mac','linux'], axis=1, inplace=True)

#Model focused on english speaking players. Remove games that do not support english.
print(f'Removing non-english supported games.... Current count: {df[~(df['supported_languages'].str.contains('english',case=False, na=False))].shape[0]}')
df.drop(df[~(df['supported_languages'].str.contains('english',case=False, na=False))].index, axis=0, inplace=True) #Drop non-english supported games
print(f'New non-english games count: {df[~(df['supported_languages'].str.contains('english',case=False, na=False))].shape[0]}')
#Remove column for supported language since only games with english support are being considered.
df.drop(['supported_languages'], axis=1, inplace=True)

print(f'Remaining Columns: {df.dtypes}') #Check remaining columns

#Define categorical and numerical columns
categorical = ["name","release_date","lead_source","employment_status","developers","categories","genres","tags"]
numerical = ["estimated_owners","required_age","price","discountdlc_count","metacritic_score","user_score","positive","negative","score_rank","achievements","recommendations","average_playtime_forever","average_playtime_two_weeks","median_playtime_forever","median_playtime_two_weeks"]
df.head()

Removing non-windows games.... Current count: 44
Updated Windows Games count: 0
Removing non-english supported games.... Current count: 13672
New non-english games count: 0
Remaining Columns: appid                             str
name                              str
release_date                      str
estimated_owners                int64
required_age                  float64
price                           int64
discountdlc_count               int64
metacritic_score                int64
user_score                      int64
positive                        int64
negative                        int64
score_rank                    float64
achievements                    int64
recommendations                 int64
average_playtime_forever        int64
average_playtime_two_weeks      int64
median_playtime_forever         int64
median_playtime_two_weeks       int64
developers                        str
categories                        str
genres                            str
tags      

,appid,name,release_date,estimated_owners,required_age,price,discountdlc_count,metacritic_score,user_score,positive,...,achievements,recommendations,average_playtime_forever,average_playtime_two_weeks,median_playtime_forever,median_playtime_two_weeks,developers,categories,genres,tags
1,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,5.24,65,0,0,0,252,...,0,231,8,0,8,0,minori,"Single-player,Steam Trading Cards,Steam Cloud,...",Adventure,"Adventure,Visual Novel,Anime,Cute"
2,Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000,0,4.99,0,0,0,0,21,...,0,0,0,0,0,0,Somer Games,"Single-player,Family Sharing",Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec..."
4,Maze Quest VR,"Apr 24, 2025",0 - 20000,0,4.99,0,0,0,0,0,...,0,0,0,0,0,0,Reality Expanded LLC,"Single-player,VR Only,Steam Leaderboards,Famil...","Action,Early Access",NaN
5,Agony VR,"Apr 5, 2023",0 - 20000,0,13.99,0,0,0,0,0,...,0,0,0,0,0,0,Ignibit,"Single-player,Tracked Controller Support,VR On...","Action,Adventure",NaN
6,Armored Brigade II,"Apr 8, 2025",0 - 20000,8,35.99,10,1,0,0,117,...,0,107,675,0,1253,0,Veitikka Studios,"Single-player,Steam Workshop,Steam Cloud,Famil...","Simulation,Strategy","Simulation,Strategy,RTS,Wargame,Grand Strategy..."


In [3]:
#------------------DATA PREPARATION AND CLEANING - CONTINUATION---------
#Check missing values
print(df.isnull().sum())

#Columns score_rank and tags have a significant number of missing values and other columns such as genre/category (for tags) and user_score/metacritic_Score can be used for the same purpose. Those columns will also be removed.
df.drop(['score_rank','tags'], axis=1, inplace=True)

#Remove rows with missing developer, category or genre (remaing of columns with NA)
df.dropna(inplace=True)

#Check missing values again
print(df.isnull().sum())

appid                              0
name                               0
release_date                       0
estimated_owners                   0
required_age                       0
price                              0
discountdlc_count                  0
metacritic_score                   0
user_score                         0
positive                           0
negative                           0
score_rank                    112101
achievements                       0
recommendations                    0
average_playtime_forever           0
average_playtime_two_weeks         0
median_playtime_forever            0
median_playtime_two_weeks          0
developers                       154
categories                       934
genres                           127
tags                           32055
dtype: int64
appid                         0
name                          0
release_date                  0
estimated_owners              0
required_age                  0
price        